# Taxi-v3 RL - Notebook demo

Doel: een korte, reproduceerbare workflow laten zien die de kern van het project aantoont.

Wat je hier ziet:
- omgeving en state-decoding
- baseline agents (random, heuristiek)
- korte Q-learning training run
- evaluatie en simpele visualisatie

## Setup
Zorg dat je dependencies zijn geinstalleerd via `pip install -r requirements.txt`.
De notebook gebruikt de code uit de map `src/` en schrijft demo-resultaten naar `results/notebook_demo`.

In [1]:
from pathlib import Path
import sys
import subprocess

# Zoek requirements.txt in huidige map of een map hoger.
req = Path("requirements.txt")
if not req.exists():
    req = Path("..") / "requirements.txt"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req)])

did_reinstall = False
try:
    import numpy as np
    _ = np.random.RandomState(0)
except Exception:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "numpy"
    ])
    did_reinstall = True

if did_reinstall:
    print("Numpy herinstallatie gedaan. Herstart de kernel en run deze cel opnieuw.")

In [4]:
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Zorg dat we vanuit de notebook de src/ modules kunnen importeren.
PROJECT_ROOT = Path('..').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.env import TaxiEnv, N_ACTIONS, N_STATES, ACTION_NAMES, decode_state
from src.agents.q_learning import QLearningAgent
from src.agents.random_agent import RandomAgent
from src.agents.heuristic_agent import HeuristicAgent

RESULTS_DIR = PROJECT_ROOT / 'results' / 'notebook_demo'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Poefj\anaconda\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Poefj\anaconda\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\Poefj\anaconda\Lib\site-packages\ipykernel\kernelapp.py", line 711, in start
    self.io_loop.start()
  File "c:\Users\Poefj\anaconda\Lib\site-packages\

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## 1) Omgeving check
We tonen een reset, een actie en de state-decoding.

In [5]:
env = TaxiEnv()
s0 = env.reset(seed=0)
decoded = decode_state(s0)

print('start state:', s0, 'decoded:', decoded)
print('action 0 =', ACTION_NAMES[0])

s1, reward, done, info = env.step(0)  # 0 = south
env.close()

print('step -> state:', s1, 'reward:', reward, 'done:', done)
print('info:', info)

NameError: name 'TaxiEnv' is not defined

## 2) Baselines (random en heuristiek)
We draaien een korte evaluatie om te laten zien dat de agents werken.

In [ ]:
def evaluate_agent(agent, episodes=50, seed=0, max_steps=200):
    env = TaxiEnv()
    returns = []
    lengths = []
    illegals = []
    success = []
    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        ep_return = 0.0
        steps = 0
        illegal = 0
        delivered = 0
        for _ in range(max_steps):
            action = agent.select_action(state, greedy=True)
            state, reward, done, info = env.step(action)
            ep_return += reward
            steps += 1
            if info.get('illegal_action'):
                illegal += 1
            if info.get('delivered'):
                delivered = 1
            if done:
                break
        returns.append(ep_return)
        lengths.append(steps)
        illegals.append(illegal)
        success.append(delivered)
    env.close()
    return {
        'return_mean': float(np.mean(returns)),
        'return_std': float(np.std(returns)),
        'len_mean': float(np.mean(lengths)),
        'illegal_mean': float(np.mean(illegals)),
        'success_rate': float(np.mean(success)),
    }

random_agent = RandomAgent(seed=0)
heuristic_agent = HeuristicAgent(seed=0)

baseline_results = {
    'random': evaluate_agent(random_agent, episodes=50, seed=0),
    'heuristic': evaluate_agent(heuristic_agent, episodes=50, seed=0),
}
baseline_results

## 3) Korte Q-learning training run
We trainen kort (klein aantal episodes) zodat dit snel draait in een demo.

In [ ]:
def train_qlearning(episodes=300, max_steps=200, seed=0, cfg=None):
    if cfg is None:
        cfg = {}
    random.seed(seed)
    np.random.seed(seed)
    env = TaxiEnv()
    agent = QLearningAgent(n_states=N_STATES, n_actions=N_ACTIONS, seed=seed, **cfg)
    log = []
    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        action = agent.select_action(state)
        ep_return = 0.0
        steps = 0
        illegal = 0
        delivered = 0

        for _ in range(max_steps):
            next_state, reward, done, info = env.step(action)
            ep_return += reward
            steps += 1
            if info.get('illegal_action'):
                illegal += 1
            if info.get('delivered'):
                delivered = 1

            agent.update(state, action, reward, next_state, None, done)
            action = agent.select_action(next_state) if not done else 0
            state = next_state
            if done:
                break

        agent.end_episode()
        log.append((ep, ep_return, steps, illegal, delivered, agent.epsilon))

    env.close()
    return agent, np.array(log, dtype=float)

cfg_path = PROJECT_ROOT / 'experiments' / 'qlearning_default.yaml'
cfg = {}
if cfg_path.exists():
    with open(cfg_path, 'r') as f:
        cfg = (yaml.safe_load(f) or {}).get('agent', {})

agent, log = train_qlearning(episodes=300, seed=0, cfg=cfg)
log[:5]

## 4) Plot training curve
We maken een eenvoudige rolling-mean plot van de returns.

In [ ]:
def smooth(x, window=20):
    if window <= 1 or len(x) < window:
        return x
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode='valid')

returns = log[:, 1]
window = 20
smoothed = smooth(returns, window=window)
x = np.arange(len(smoothed)) + window

plt.figure(figsize=(8, 4))
plt.plot(x, smoothed, label='qlearning (smoothed)')
plt.xlabel('episode')
plt.ylabel(f'return (rolling mean, w={window})')
plt.title('training curve (demo run)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

## 5) Evaluatie van de getrainde agent
We zetten epsilon naar 0 zodat de policy greedy is tijdens evaluatie.

In [ ]:
agent.epsilon = 0.0
trained_stats = evaluate_agent(agent, episodes=50, seed=123)
trained_stats

## 6) Policy visualisatie
We gebruiken het bestaande script `src.plot_policy` om een grid te renderen.

In [ ]:
import subprocess

weights_path = RESULTS_DIR / 'qtable.npy'
agent.save(str(weights_path))

policy_path = RESULTS_DIR / 'policy.png'
subprocess.run(
    [sys.executable, '-m', 'src.plot_policy',
     '--weights', str(weights_path),
     '--output', str(policy_path),
     '--title', 'Q-learning policy (demo)'],
    check=True,
)

img = plt.imread(policy_path)
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')